# Proyecto AEMET - Despliegue en AWS EC2, NLP y FastAPI

Este notebook resume y explica el trabajo de integración y despliegue del proyecto. 
La meta de esta fase ha sido **coger todas las piezas individuales** (el ETL, el frontend interactivo de Tamara, los modelos de NLP y XGBoost) y **conectarlas en una arquitectura robusta** desplegada en una máquina virtual de la nube (AWS EC2).

## 1. Organización del Código en la Nube (Carpeta `Despliegue_EC2`)

Para que se pueda evaluar exactamente qué código está corriendo en nuestro servidor AWS EC2, he copiado los archivos de producción dentro de la carpeta `Despliegue_EC2/` de este directorio.

Los archivos y carpetas clave son:
- `api_unificada.py`: El backend en FastAPI que centraliza las conexiones a PostgreSQL RDS, sirve los modelos de Machine Learning (XGBoost) y aloja la lógica del LLM (Gemini) para Text-to-SQL.
- `app_unificada.py`: La interfaz gráfica en Streamlit (basada en el trabajo de Tamara), adaptada para no conectarse a la base de datos directamente, sino realizar peticiones HTTP a la API.
- `modelos_max/` y `modelos_min/`: Directorios con los modelos XGBoost preentrenados (`.pkl`) listos para inferencia en producción.
- `entrenamiento_modelos.py`: Script original encargado de generar dichos modelos.
- `conectar_ec2.sh`: Script de bash para automatizar el acceso por SSH a la instancia de AWS.
- `run_api.py` / `run_app.py`: Scripts lanzadores para mantener los servidores vivos.

## 2. Arquitectura Final Implementada

El flujo de datos está diseñado para ser completamente escalable, seguro y profesional. La separación de responsabilidades (Frontend vs Backend) permite que la web sea muy ligera.

```mermaid
graph LR
    A[Usuario Final] -->|Interactúa (Puerto 8501)| B(Streamlit - app_unificada)
    B -->|HTTP GET/POST (Puerto 8000)| C{FastAPI Backend - api_unificada}
    C -->|Consultas SQL (Puerto 5432)| D[(AWS RDS PostgreSQL)]
    C -->|API REST| E[Google Gemini LLM]
    C -->|Inferencia| F[Modelos ML Locales]
```

## 3. Demostración del Motor NLP (Text-to-SQL)

Uno de mis principales desarrollos ha sido la integración de un modelo de Lenguaje Natural (Google Gemini) para que actúe como un analista de datos virtual. 
A través de LangChain y Pydantic, el backend recibe una pregunta en lenguaje natural, genera el SQL adaptado estrictamente a nuestro esquema DDL y devuelve los registros al Frontend.

In [ ]:
# Importamos librerías visuales y de red para la demostración
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid", palette="pastel")
plt.rcParams['figure.figsize'] = (10, 5)

print("Librerías de análisis cargadas.")

In [ ]:
# Apuntamos a la IP pública de nuestra instancia EC2 en AWS
FASTAPI_URL = "http://18.198.208.67:8000"  

pregunta = "¿Cuáles fueron las 5 temperaturas máximas más altas registradas en Madrid durante el verano de 2023?"
print(f"Pregunta enviada desde el Frontend: '{pregunta}'\n")

try:
    # Simulamos lo que hace Streamlit por debajo
    respuesta = requests.post(f"{FASTAPI_URL}/ask", json={"pregunta": pregunta}, timeout=15)
    
    if respuesta.status_code == 200:
        datos = respuesta.json()
        print("Traducción a SQL y consulta a BD completadas de forma transparente.\n")
        print(f"SQL Generado internamente:\n{datos.get('sql_generado')}\n")
        
        df_nlp = pd.DataFrame(datos.get('datos', []))
        print("Resultados parseados listos para mostrar al usuario:")
        display(df_nlp.head())
    else:
        print(f"Error de la API: {respuesta.status_code} - {respuesta.text}")
except Exception as e:
    print("No se pudo conectar a la API en EC2. Asegúrate de que la instancia esté encendida.")
    print(f"Detalle: {e}")

## 4. Consumo de la API para Históricos e Inferencia ML

La segunda responsabilidad de la API es servir los modelos predictivos que el equipo de Data Science preparó, así como devolver el histórico de forma rápida para visualizaciones dinámicas.

In [ ]:
idema_test = "3195" # Estación Madrid, Retiro

try:
    print("Obteniendo histórico mensual para la visualización de Streamlit...")
    res_hist = requests.get(f"{FASTAPI_URL}/historico/obtener_historico", 
                            params={"id": idema_test, "fecha_inicio": "2024-01-01", "fecha_fin": "2024-01-31"})
    
    if res_hist.status_code == 200:
        df_hist = pd.DataFrame(res_hist.json().get("registros", []))
        df_hist['fecha'] = pd.to_datetime(df_hist['fecha'])
        
        plt.figure(figsize=(12, 5))
        sns.lineplot(data=df_hist, x='fecha', y='tmax', marker="o", label="Temp. Máxima", color="#ff6b6b")
        sns.lineplot(data=df_hist, x='fecha', y='tmed', marker="s", label="Temp. Media", color="#4ecdc4")
        sns.lineplot(data=df_hist, x='fecha', y='tmin', marker="^", label="Temp. Mínima", color="#45b7d1")
        
        plt.title(f"Evolución Térmica (API) - Estación {idema_test}", fontsize=14, pad=15)
        plt.xlabel("Fecha")
        plt.ylabel("Temperatura (ºC)")
        plt.legend()
        plt.show()
        
        print("\nPidiendo inferencia en tiempo real al modelo de Machine Learning...")
        res_ult = requests.get(f"{FASTAPI_URL}/historico/ultima_tmed", params={"id": idema_test})
        if res_ult.status_code == 200:
            ultima_tmed = res_ult.json()['tmed']
            res_pred = requests.post(f"{FASTAPI_URL}/modelos_max/{idema_test}/predict", json={"features": [ultima_tmed]})
            
            if res_pred.status_code == 200:
                prediccion = res_pred.json()['prediccion'][0][0]
                print(f"-> Con una T. Media actual de {ultima_tmed}ºC, FastAPI predice una T. Máxima de {prediccion:.2f}ºC")
except Exception as e:
    print(f"Error de conexión: {e}")

## 5. Conclusión de la Integración

El principal reto de esta fase ha sido **desacoplar el frontend del acceso a datos**. Inicialmente, los notebooks y prototipos de mis compañeros accedían directamente a la base de datos o ejecutaban los modelos en memoria.

Al implementar **FastAPI** como Middleware alojado en **EC2**:
1. Centralizamos la seguridad (las credenciales de RDS o Gemini solo están en el servidor backend).
2. Mejoramos el rendimiento (FastAPI carga los modelos `.pkl` en memoria una sola vez al arrancar).
3. Permitimos que el frontend de **Streamlit** (que hereda el buen diseño de Tamara) sea muy ligero y rápido, limitándose a pintar datos y gráficas a partir de las respuestas HTTP.

*(Revisa el código completo que se encuentra en la nube accediendo a la carpeta `Despliegue_EC2/`)*.